# Run Real Video VAE Latent Probe

本文件是 `real_video_vae_latent_probe` 运行入口。消费上游 processed dataset handoff 与 session-only VAE 模型, 并把 runner、checker、table rebuild 与结果打包全部委托给仓库模块。

## 00 Runtime Identity And User Config

集中声明本次 probe run 的运行模式、family 身份、processed dataset key、仓库根目录、session local 路径、模型来源与 formal 门禁开关。

紧随其后的代码单元是集中手动配置区；优先级为：显式环境变量 > 该单元中的手动配置。

仓库 bootstrap 相关变量在该配置单元内解析为唯一来源；后续 bootstrap 单元只消费已解析的 `TSTW_REPO_URL`、`TSTW_REPO_BRANCH` 与 `TSTW_REPO_ROOT`，若未先执行配置单元将显式报错，而不是静默回退到另一套默认值。

`REPO_URL` 与 `REPO_ROOT` 控制 Colab 冷启动时的仓库 bootstrap；
`RUN_MODE` 与 `RUNTIME_PROFILE` 控制运行模式；
`TSTW_NOTEBOOK_RUN_PRESET=colab_result_small` 会默认启用小样本 Colab 结果模式；
`TSTW_NOTEBOOK_RUN_PRESET=formal_full` 可切回完整 formal 覆盖；
`WORKFLOW_KEY`、`STEP_KEY`、`FAMILY_ID` 用于结果包登记；
`PROCESSED_DATASET_KEY` 来自上游 processed dataset notebook 的最终 handoff；
`PROCESSED_DATASET_MANIFEST` 指向 session local 的 manifest；
`MODEL_ID` 指定 session-only 模型来源；
`REQUIRE_FORMAL_PASS=True` 表示正式检查失败时必须阻断打包。

In [ ]:
import os

NOTEBOOK_MANUAL_CONFIG = {
    'repo_url': 'https://github.com/RICHAAARC/TSTW-VW.git',  # Colab 冷启动时使用的仓库 clone URL
    'repo_branch': 'main',  # Colab 冷启动时使用的仓库分支
    'repo_root': '/content/TSTW-VW',  # Colab 本地仓库根目录
    'notebook_run_preset': 'formal_full',  # 选择 notebook 预设运行规模
    'execution_runtime_profile': 'l4_formal',  # 执行侧 runtime profile 名称
    'workflow_key': 'real_video_vae_latent_probe_completion_formal',  # 结果登记使用的 workflow 标识
    'step_key': 'step02_run_real_video_vae_latent_probe',  # 结果登记使用的 step 标识
    'family_id_template': 'real_video_vae_latent_probe__formal__davis2017_trainval480p__utc_time__short_commit',  # family_id 的模板字符串
    'processed_dataset_key': 'real_video_vae_latent_probe__davis2017_trainval480p__256x256__32f__8fps__freeze001',  # processed dataset 的 registry key
    'drive_root': '/content/drive/MyDrive',  # Drive 根目录
    'local_runtime_root': '/content/TSTW_runtime',  # session local 运行根目录
    'model_id': 'stabilityai/sd-vae-ft-mse',  # session-only VAE 模型来源
    'batch_size_frames': 512,  # VAE encode/decode 的单次 frame batch 大小
    # Governed default anchor: 'cross_event_vae_batching_enabled': False，表示默认关闭跨 event 的 VAE batching。
    'cross_event_vae_batching_enabled': True,  # 是否启用跨 event 的 VAE batching
    'cross_event_vae_decode_batch_size': 64,  # 跨 event decode 的聚合 batch 大小
    'cross_event_vae_encode_batch_size': 64,  # 跨 event encode 的聚合 batch 大小
    'shard_count': 1,  # 外层 event shard 总数
    'shard_index': 0,  # 当前运行选择的外层 shard 编号
    'worker_count': 1,  # 已选 shard 内部的本地 worker 数
    'samples_per_role_override': None,  # 覆盖每个 split-role pair 的样本数；None 表示使用 protocol 默认值
    'require_lpips_evidence': True,  # 是否要求 LPIPS 证据
    'enable_clip_similarity': False,  # 是否启用 CLIP similarity 指标
    'enable_motion_consistency': True,  # 是否启用 motion consistency 指标
    'allow_inconclusive_review': False,  # 是否允许 INCONCLUSIVE 结果继续审查
    'run_main_formal': False,  # 是否执行主 formal runner
    'run_stage2_mechanism_calibration': True,  # 是否执行 mechanism calibration
    'run_stage2_local_clip_sync_forensics': False,  # tl02 validation 默认关闭 local_clip forensics；需要时再显式打开
    'package_non_formal_audit_bundle': False,  # tl02 validation 默认不打 non-formal audit bundle；需要时再显式打开
    'require_stage2_mechanism_pass': False,  # 是否要求 mechanism audit 为 PASS
    'stage2_calibration_target': 'tl02_controlled_validation',  # default_search 或 tl02_controlled_validation
}

NOTEBOOK_MANUAL_ENV_MAP = {
    'repo_url': 'TSTW_REPO_URL',
    'repo_branch': 'TSTW_REPO_BRANCH',
    'repo_root': 'TSTW_REPO_ROOT',
    'notebook_run_preset': 'TSTW_NOTEBOOK_RUN_PRESET',
    'execution_runtime_profile': 'TSTW_EXECUTION_RUNTIME_PROFILE',
    'workflow_key': 'TSTW_WORKFLOW_KEY',
    'step_key': 'TSTW_STEP_KEY',
    'family_id_template': 'TSTW_FAMILY_ID_TEMPLATE',
    'processed_dataset_key': 'TSTW_PROCESSED_DATASET_KEY',
    'drive_root': 'TSTW_DRIVE_ROOT',
    'local_runtime_root': 'TSTW_LOCAL_RUNTIME_ROOT',
    'model_id': 'TSTW_MODEL_ID',
    'batch_size_frames': 'TSTW_BATCH_SIZE_FRAMES',
    'cross_event_vae_batching_enabled': 'TSTW_CROSS_EVENT_VAE_BATCHING_ENABLED',
    'cross_event_vae_decode_batch_size': 'TSTW_CROSS_EVENT_VAE_DECODE_BATCH_SIZE',
    'cross_event_vae_encode_batch_size': 'TSTW_CROSS_EVENT_VAE_ENCODE_BATCH_SIZE',
    'shard_count': 'TSTW_SHARD_COUNT',
    'shard_index': 'TSTW_SHARD_INDEX',
    'worker_count': 'TSTW_WORKER_COUNT',
    'samples_per_role_override': 'TSTW_SAMPLES_PER_ROLE_OVERRIDE',
    'require_lpips_evidence': 'TSTW_REQUIRE_LPIPS_EVIDENCE',
    'enable_clip_similarity': 'TSTW_ENABLE_CLIP_SIMILARITY',
    'enable_motion_consistency': 'TSTW_ENABLE_MOTION_CONSISTENCY',
    'allow_inconclusive_review': 'TSTW_ALLOW_INCONCLUSIVE_REVIEW',
    'run_main_formal': 'TSTW_RUN_MAIN_FORMAL',
    'run_stage2_mechanism_calibration': 'TSTW_RUN_STAGE2_MECHANISM_CALIBRATION',
    'run_stage2_local_clip_sync_forensics': 'TSTW_RUN_STAGE2_LOCAL_CLIP_SYNC_FORENSICS',
    'package_non_formal_audit_bundle': 'TSTW_PACKAGE_NON_FORMAL_AUDIT_BUNDLE',
    'require_stage2_mechanism_pass': 'TSTW_REQUIRE_STAGE2_MECHANISM_PASS',
    'stage2_calibration_target': 'TSTW_STAGE2_CALIBRATION_TARGET',
}

for config_key, env_key in NOTEBOOK_MANUAL_ENV_MAP.items():
    env_value = os.environ.get(env_key)
    if env_value is not None and str(env_value).strip() != '':
        continue
    config_value = NOTEBOOK_MANUAL_CONFIG.get(config_key)
    if config_value is None:
        continue
    if isinstance(config_value, bool):
        os.environ[env_key] = '1' if config_value else '0'
    else:
        os.environ[env_key] = str(config_value)

print({
    'manual_config_keys': list(NOTEBOOK_MANUAL_CONFIG.keys()),
    'active_manual_overrides': {
        config_key: NOTEBOOK_MANUAL_CONFIG[config_key]
        for config_key in NOTEBOOK_MANUAL_CONFIG
        if NOTEBOOK_MANUAL_CONFIG[config_key] is not None
    },
})

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

def _resolve_required_environment_text(env_key):
    raw_value = os.environ.get(env_key)
    if raw_value is None:
        raise RuntimeError(
            f'{env_key} is not set; run Cell 3 first or define the variable before running this bootstrap cell'
        )
    resolved_value = str(raw_value).strip()
    if not resolved_value:
        raise RuntimeError(
            f'{env_key} is empty; update NOTEBOOK_MANUAL_CONFIG in Cell 3 or define the variable explicitly before running this bootstrap cell'
        )
    return resolved_value

def _format_process_output(output_text):
    resolved_output = output_text.strip()
    return resolved_output if resolved_output else '<empty>'

def _describe_repo_root_entries(repo_root):
    try:
        entry_names = sorted(path.name for path in repo_root.iterdir())
    except OSError as exc:
        return [f'<failed_to_list_entries: {exc}>']
    preview_limit = 10
    if len(entry_names) <= preview_limit:
        return entry_names
    return entry_names[:preview_limit] + ['<truncated>']

def _bootstrap_repository_root(repo_root, repo_url, repo_branch):
    repo_package_marker = repo_root / 'paper_workflow' / '__init__.py'
    repo_git_dir = repo_root / '.git'
    if repo_package_marker.exists():
        if not repo_git_dir.exists():
            raise RuntimeError(
                'repository bootstrap found the governed notebook package marker but the git metadata directory is missing.\n'
                f'TSTW_REPO_ROOT={repo_root}\n'
                f'expected_git_dir={repo_git_dir}\n'
                'Remove the incomplete checkout or point TSTW_REPO_ROOT to a valid git clone before rerunning Cell 4.'
            )
        fetch_result = subprocess.run(
            ['git', 'fetch', '--depth', '1', 'origin', repo_branch],
            check=False,
            capture_output=True,
            text=True,
            cwd=repo_root,
        )
        if fetch_result.returncode != 0:
            raise RuntimeError(
                'repository bootstrap failed during git fetch for an existing checkout.\n'
                f'TSTW_REPO_BRANCH={repo_branch}\n'
                f'TSTW_REPO_ROOT={repo_root}\n'
                f'git_fetch_returncode={fetch_result.returncode}\n'
                f'git_fetch_stdout={_format_process_output(fetch_result.stdout)}\n'
                f'git_fetch_stderr={_format_process_output(fetch_result.stderr)}\n'
                'Resolve network or authentication issues, or remove the stale checkout before rerunning Cell 4.'
            )
        checkout_result = subprocess.run(
            ['git', 'checkout', repo_branch],
            check=False,
            capture_output=True,
            text=True,
            cwd=repo_root,
        )
        if checkout_result.returncode != 0:
            checkout_result = subprocess.run(
                ['git', 'checkout', '-b', repo_branch, '--track', f'origin/{repo_branch}'],
                check=False,
                capture_output=True,
                text=True,
                cwd=repo_root,
            )
        if checkout_result.returncode != 0:
            raise RuntimeError(
                'repository bootstrap failed while checking out the requested branch in an existing checkout.\n'
                f'TSTW_REPO_BRANCH={repo_branch}\n'
                f'TSTW_REPO_ROOT={repo_root}\n'
                f'git_checkout_returncode={checkout_result.returncode}\n'
                f'git_checkout_stdout={_format_process_output(checkout_result.stdout)}\n'
                f'git_checkout_stderr={_format_process_output(checkout_result.stderr)}\n'
                'Commit or discard local changes in the existing checkout before rerunning Cell 4.'
            )
        pull_result = subprocess.run(
            ['git', 'pull', '--ff-only', 'origin', repo_branch],
            check=False,
            capture_output=True,
            text=True,
            cwd=repo_root,
        )
        if pull_result.returncode != 0:
            raise RuntimeError(
                'repository bootstrap failed while fast-forwarding the existing checkout.\n'
                f'TSTW_REPO_BRANCH={repo_branch}\n'
                f'TSTW_REPO_ROOT={repo_root}\n'
                f'git_pull_returncode={pull_result.returncode}\n'
                f'git_pull_stdout={_format_process_output(pull_result.stdout)}\n'
                f'git_pull_stderr={_format_process_output(pull_result.stderr)}\n'
                'Commit or discard local changes, or recreate the checkout before rerunning Cell 4.'
            )
        return repo_package_marker
    if repo_root.exists():
        if repo_root.is_file():
            raise RuntimeError(
                'repository bootstrap failed because TSTW_REPO_ROOT points to a file instead of a repository directory.\n'
                f'TSTW_REPO_ROOT={repo_root}\n'
                'Update TSTW_REPO_ROOT to a directory path before rerunning Cell 4.'
            )
        repo_root_entries = _describe_repo_root_entries(repo_root)
        if repo_root_entries:
            if repo_git_dir.exists():
                raise RuntimeError(
                    'repository bootstrap failed because TSTW_REPO_ROOT points to a git repository that does not contain the governed notebook package marker.\n'
                    f'TSTW_REPO_ROOT={repo_root}\n'
                    f'expected_marker={repo_package_marker}\n'
                    f'root_entries={repo_root_entries}\n'
                    'Update TSTW_REPO_ROOT to the cloned TSTW-VW repository root, or remove the incomplete checkout before rerunning Cell 4.'
                )
            raise RuntimeError(
                'repository bootstrap failed because TSTW_REPO_ROOT already exists and is not empty.\n'
                f'TSTW_REPO_ROOT={repo_root}\n'
                f'expected_marker={repo_package_marker}\n'
                f'root_entries={repo_root_entries}\n'
                'Set TSTW_REPO_ROOT to an empty directory or pre-clone the repository into that path before rerunning Cell 4.'
            )
    repo_root.parent.mkdir(parents=True, exist_ok=True)
    clone_result = subprocess.run(
        [
            'git',
            'clone',
            '--depth',
            '1',
            '--branch',
            repo_branch,
            repo_url,
            str(repo_root),
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if clone_result.returncode != 0:
        raise RuntimeError(
            'repository bootstrap failed during git clone.\n'
            f'TSTW_REPO_URL={repo_url}\n'
            f'TSTW_REPO_BRANCH={repo_branch}\n'
            f'TSTW_REPO_ROOT={repo_root}\n'
            f'git_clone_returncode={clone_result.returncode}\n'
            f'git_clone_stdout={_format_process_output(clone_result.stdout)}\n'
            f'git_clone_stderr={_format_process_output(clone_result.stderr)}\n'
            'Set TSTW_REPO_URL to an authenticated clone URL if the repository requires authentication in the current environment, or pre-clone the repository into TSTW_REPO_ROOT before rerunning Cell 4.'
        )
    if not repo_package_marker.exists():
        raise RuntimeError(
            'repository bootstrap finished the clone command but the governed notebook package marker is still missing.\n'
            f'TSTW_REPO_ROOT={repo_root}\n'
            f'expected_marker={repo_package_marker}\n'
            'Inspect the clone target and rerun Cell 4 after the repository layout is corrected.'
        )
    return repo_package_marker

REPO_URL = _resolve_required_environment_text('TSTW_REPO_URL')
REPO_BRANCH = _resolve_required_environment_text('TSTW_REPO_BRANCH')
REPO_ROOT = Path(_resolve_required_environment_text('TSTW_REPO_ROOT')).expanduser()
os.environ['TSTW_REPO_URL'] = REPO_URL
os.environ['TSTW_REPO_BRANCH'] = REPO_BRANCH
os.environ['TSTW_REPO_ROOT'] = str(REPO_ROOT)
REPO_PACKAGE_MARKER = _bootstrap_repository_root(
    repo_root=REPO_ROOT,
    repo_url=REPO_URL,
    repo_branch=REPO_BRANCH,
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from paper_workflow.notebook_utils import real_video_vae_latent_probe_workflow as probe_workflow
from paper_workflow.notebook_utils import run_timing_workflow
from paper_workflow.notebook_utils import runtime_profile_workflow

RUN_MODE = 'formal'
RUNTIME_PROFILE = 'formal'
NOTEBOOK_RUN_PRESET = os.environ.get('TSTW_NOTEBOOK_RUN_PRESET', 'colab_result_small').strip() or 'colab_result_small'
if NOTEBOOK_RUN_PRESET not in {'colab_result_small', 'formal_full'}:
    raise ValueError('TSTW_NOTEBOOK_RUN_PRESET must be one of: colab_result_small, formal_full')
if NOTEBOOK_RUN_PRESET == 'colab_result_small':
    os.environ.setdefault('TSTW_EXECUTION_RUNTIME_PROFILE', 'l4_debug')
    os.environ.setdefault('TSTW_SAMPLES_PER_ROLE_OVERRIDE', '1')
    os.environ.setdefault('TSTW_BATCH_SIZE_FRAMES', '4')
    os.environ.setdefault('TSTW_ALLOW_INCONCLUSIVE_REVIEW', '1')
    os.environ.setdefault('TSTW_REQUIRE_LPIPS_EVIDENCE', '0')
    os.environ.setdefault('TSTW_ENABLE_CLIP_SIMILARITY', '0')
    os.environ.setdefault('TSTW_ENABLE_MOTION_CONSISTENCY', '1')
EXECUTION_RUNTIME_PROFILE = os.environ.get('TSTW_EXECUTION_RUNTIME_PROFILE', 'l4_formal')
PROFILE_RUNTIME = RUN_MODE == 'formal'
PROFILE_GPU_RUNTIME = PROFILE_RUNTIME
GPU_PROFILE_INTERVAL_SECONDS = 2
PROFILE_DRIVE_IO = PROFILE_RUNTIME
DRIVE_IO_SAMPLE_SIZE_MB = 64
WRITE_RUNTIME_RECOMMENDATION = PROFILE_RUNTIME
WORKFLOW_KEY = os.environ.get('TSTW_WORKFLOW_KEY', 'real_video_vae_latent_probe_completion_formal')
STEP_KEY = os.environ.get('TSTW_STEP_KEY', 'step02_run_real_video_vae_latent_probe')
FAMILY_ID_TEMPLATE = os.environ.get(
    'TSTW_FAMILY_ID_TEMPLATE',
    'real_video_vae_latent_probe__formal__davis2017_trainval480p__utc_time__short_commit',
)
PROCESSED_DATASET_KEY = os.environ.get(
    'TSTW_PROCESSED_DATASET_KEY',
    'real_video_vae_latent_probe__davis2017_trainval480p__256x256__32f__8fps__freeze001',
)
DRIVE_ROOT = Path(os.environ.get('TSTW_DRIVE_ROOT', '/content/drive/MyDrive'))
LOCAL_RUNTIME_ROOT = Path(os.environ.get('TSTW_LOCAL_RUNTIME_ROOT', '/content/TSTW_runtime'))
DEFAULT_LPIPS_MODEL_ROOT = LOCAL_RUNTIME_ROOT / 'session_models' / 'lpips'
BATCH_SIZE_FRAMES = 8
PROTOCOL_CONFIG_PATH = Path('configs/protocol/real_video_vae_latent_probe.json')
EXECUTION_RUNTIME_PROFILE_CONFIG = runtime_profile_workflow.load_runtime_profile_config(
    runtime_profile=EXECUTION_RUNTIME_PROFILE,
)
PROFILE_RUNTIME = bool(EXECUTION_RUNTIME_PROFILE_CONFIG['profile_runtime'])
PROFILE_GPU_RUNTIME = bool(EXECUTION_RUNTIME_PROFILE_CONFIG['profile_gpu_runtime'])
GPU_PROFILE_INTERVAL_SECONDS = int(EXECUTION_RUNTIME_PROFILE_CONFIG['gpu_profile_interval_seconds'])
PROFILE_DRIVE_IO = bool(EXECUTION_RUNTIME_PROFILE_CONFIG['profile_drive_io'])
DRIVE_IO_SAMPLE_SIZE_MB = int(EXECUTION_RUNTIME_PROFILE_CONFIG['drive_io_sample_size_mb'])
WRITE_RUNTIME_RECOMMENDATION = bool(EXECUTION_RUNTIME_PROFILE_CONFIG['write_runtime_recommendation'])
default_batch_size_frames = int(EXECUTION_RUNTIME_PROFILE_CONFIG['vae_batch_size_frames'])
LPIPS_BATCH_SIZE = int(EXECUTION_RUNTIME_PROFILE_CONFIG['lpips_batch_size'])
CLIP_BATCH_SIZE = int(EXECUTION_RUNTIME_PROFILE_CONFIG['clip_batch_size'])
WORKER_COUNT = int(EXECUTION_RUNTIME_PROFILE_CONFIG['worker_count'])
VIDEO_IO_WORKER_COUNT = int(EXECUTION_RUNTIME_PROFILE_CONFIG['video_io_worker_count'])
ATTACK_WORKER_COUNT = int(EXECUTION_RUNTIME_PROFILE_CONFIG['attack_worker_count'])
SHARD_COUNT = int(EXECUTION_RUNTIME_PROFILE_CONFIG['shard_count'])
REUSE_ENCODED_LATENTS = bool(EXECUTION_RUNTIME_PROFILE_CONFIG['reuse_encoded_latents'])
REUSE_DECODED_VIDEOS = bool(EXECUTION_RUNTIME_PROFILE_CONFIG['reuse_decoded_videos'])
REUSE_ATTACKED_VIDEOS = bool(EXECUTION_RUNTIME_PROFILE_CONFIG['reuse_attacked_videos'])
# 直接修改 BATCH_SIZE_FRAMES 可控制 VAE encode/decode 单次送入 GPU 的 frame 数；若设置 TSTW_BATCH_SIZE_FRAMES，则环境变量优先。
batch_size_frames_text = os.environ.get('TSTW_BATCH_SIZE_FRAMES', '').strip()
resolved_batch_size_frames = default_batch_size_frames
if batch_size_frames_text:
    try:
        resolved_batch_size_frames = int(batch_size_frames_text)
    except ValueError as exc:
        raise ValueError('TSTW_BATCH_SIZE_FRAMES must be a positive integer') from exc
    if resolved_batch_size_frames < 1:
        raise ValueError('TSTW_BATCH_SIZE_FRAMES must be a positive integer')
BATCH_SIZE_FRAMES = resolved_batch_size_frames

def _resolve_bool_environment_value(env_key, default_value):
    raw_value = os.environ.get(env_key, '').strip().lower()
    if raw_value in {'1', 'true', 'yes', 'on'}:
        return True
    if raw_value in {'0', 'false', 'no', 'off'}:
        return False
    if raw_value:
        raise ValueError(f'{env_key} must be a boolean value')
    return bool(default_value)

def _resolve_positive_int_environment_value(env_key, default_value):
    raw_value = os.environ.get(env_key, '').strip()
    if not raw_value:
        return int(default_value)
    try:
        resolved_value = int(raw_value)
    except ValueError as exc:
        raise ValueError(f'{env_key} must be a positive integer') from exc
    if resolved_value < 1:
        raise ValueError(f'{env_key} must be a positive integer')
    return resolved_value

CROSS_EVENT_VAE_BATCHING_ENABLED = _resolve_bool_environment_value(
    'TSTW_CROSS_EVENT_VAE_BATCHING_ENABLED',
    False,
)
CROSS_EVENT_VAE_DECODE_BATCH_SIZE = _resolve_positive_int_environment_value(
    'TSTW_CROSS_EVENT_VAE_DECODE_BATCH_SIZE',
    4,
)
CROSS_EVENT_VAE_ENCODE_BATCH_SIZE = _resolve_positive_int_environment_value(
    'TSTW_CROSS_EVENT_VAE_ENCODE_BATCH_SIZE',
    4,
)
shard_count_text = os.environ.get('TSTW_SHARD_COUNT', '').strip()
resolved_shard_count = SHARD_COUNT
if shard_count_text:
    try:
        resolved_shard_count = int(shard_count_text)
    except ValueError as exc:
        raise ValueError('TSTW_SHARD_COUNT must be a positive integer') from exc
    if resolved_shard_count < 1:
        raise ValueError('TSTW_SHARD_COUNT must be a positive integer')
SHARD_COUNT = resolved_shard_count
shard_index_text = os.environ.get('TSTW_SHARD_INDEX', '').strip()
resolved_shard_index = 0
if shard_index_text:
    try:
        resolved_shard_index = int(shard_index_text)
    except ValueError as exc:
        raise ValueError('TSTW_SHARD_INDEX must be a non-negative integer') from exc
    if resolved_shard_index < 0:
        raise ValueError('TSTW_SHARD_INDEX must be a non-negative integer')
if resolved_shard_index >= SHARD_COUNT:
    raise ValueError('TSTW_SHARD_INDEX must satisfy 0 <= TSTW_SHARD_INDEX < TSTW_SHARD_COUNT')
SHARD_INDEX = resolved_shard_index
worker_count_text = os.environ.get('TSTW_WORKER_COUNT', '').strip()
resolved_worker_count = WORKER_COUNT
if worker_count_text:
    try:
        resolved_worker_count = int(worker_count_text)
    except ValueError as exc:
        raise ValueError('TSTW_WORKER_COUNT must be a positive integer') from exc
    if resolved_worker_count < 1:
        raise ValueError('TSTW_WORKER_COUNT must be a positive integer')
WORKER_COUNT = resolved_worker_count
if CROSS_EVENT_VAE_BATCHING_ENABLED and WORKER_COUNT != 1:
    raise ValueError('cross-event VAE batching requires TSTW_WORKER_COUNT=1 in the first governed implementation')
LPIPS_MODEL_ROOT_TEXT = (
    os.environ.get('TSTW_LPIPS_MODEL_ROOT')
    or os.environ.get('LPIPS_MODEL_ROOT')
    or ''
).strip()
REQUIRE_LPIPS_EVIDENCE = os.environ.get(
    'TSTW_REQUIRE_LPIPS_EVIDENCE',
    '1' if RUN_MODE == 'formal' else '0',
) == '1'
LPIPS_MODEL_ROOT_PATH = Path(LPIPS_MODEL_ROOT_TEXT) if LPIPS_MODEL_ROOT_TEXT else DEFAULT_LPIPS_MODEL_ROOT
LPIPS_MODEL_ROOT = str(LPIPS_MODEL_ROOT_PATH)
ENABLE_LPIPS = REQUIRE_LPIPS_EVIDENCE or bool(LPIPS_MODEL_ROOT_TEXT)
LPIPS_CACHE_MARKER_PATH = LPIPS_MODEL_ROOT_PATH / 'session_local_lpips_cache_root.txt'
ENABLE_CLIP_SIMILARITY = os.environ.get('TSTW_ENABLE_CLIP_SIMILARITY', '0') == '1'
ENABLE_MOTION_CONSISTENCY = os.environ.get('TSTW_ENABLE_MOTION_CONSISTENCY', '1') == '1'
runner_samples_per_role_override_text = os.environ.get('TSTW_SAMPLES_PER_ROLE_OVERRIDE', '').strip()
# 直接修改 RUNNER_SAMPLES_PER_ROLE_OVERRIDE 或设置 TSTW_SAMPLES_PER_ROLE_OVERRIDE，可放大或缩小本轮 runner 的样本覆盖规模。
resolved_runner_samples_per_role_override = None
if runner_samples_per_role_override_text:
    try:
        resolved_runner_samples_per_role_override = int(runner_samples_per_role_override_text)
    except ValueError as exc:
        raise ValueError('TSTW_SAMPLES_PER_ROLE_OVERRIDE must be a positive integer') from exc
    if resolved_runner_samples_per_role_override < 1:
        raise ValueError('TSTW_SAMPLES_PER_ROLE_OVERRIDE must be a positive integer')
RUNNER_SAMPLES_PER_ROLE_OVERRIDE = resolved_runner_samples_per_role_override
REQUIRE_STAGE2_MECHANISM_PASS = os.environ.get('TSTW_REQUIRE_STAGE2_MECHANISM_PASS', '0') == '1'
ENABLE_STAGE2_MECHANISM_CALIBRATION = os.environ.get('TSTW_RUN_STAGE2_MECHANISM_CALIBRATION', '0') == '1'
GIT_COMMIT = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
FAMILY_ID = probe_workflow.materialize_family_id(
    family_id_template=FAMILY_ID_TEMPLATE,
    git_commit=GIT_COMMIT,
)
PROCESSED_DATASET_ROOT = DRIVE_ROOT / 'TSTW' / 'datasets' / 'processed' / PROCESSED_DATASET_KEY
LOCAL_DATASET_ROOT = LOCAL_RUNTIME_ROOT / 'datasets' / PROCESSED_DATASET_KEY
PROCESSED_DATASET_MANIFEST = LOCAL_DATASET_ROOT / 'dataset_manifest.json'
LOCAL_MODEL_ROOT = LOCAL_RUNTIME_ROOT / 'session_models' / 'autoencoder_kl'
FAMILY_ROOT = DRIVE_ROOT / 'TSTW' / 'results' / 'families' / FAMILY_ID
RUN_ROOT = LOCAL_RUNTIME_ROOT / 'runs' / 'real_video_vae_latent_probe_formal'
RUN_ID = RUN_ROOT.name
RUNTIME_PROFILE_ROOT = RUN_ROOT / 'runtime_profile'
RUNTIME_CONFIG_PATH = RUN_ROOT / 'artifacts' / 'runtime_config.json'
SESSION_MODEL_MANIFEST_PATH = RUN_ROOT / 'artifacts' / 'session_model_manifest.json'
FORMAL_VALIDATION_SUMMARY_PATH = RUNTIME_PROFILE_ROOT / 'formal_validation_summary.json'
STAGE2_MECHANISM_CONFIG_PATH = Path('configs/protocol/stage2_mechanism_gate.json')
STAGE2_MECHANISM_SUMMARY_PATH = RUN_ROOT / 'artifacts' / 'stage2_mechanism_decision.json'
STAGE2_MECHANISM_CALIBRATION_GRID_PATH = Path('configs/ablation/stage2_vae_mechanism_calibration_grid.json')
STAGE2_MECHANISM_CALIBRATION_RUN_ROOT = LOCAL_RUNTIME_ROOT / 'runs' / 'real_video_vae_latent_probe_stage2_mechanism_calibration'
STAGE2_MECHANISM_CANDIDATE_METHOD_CONFIG_PATH = STAGE2_MECHANISM_CALIBRATION_RUN_ROOT / 'artifacts' / 'tubelet_sync_real_video_vae_candidate.json'
STAGE2_MECHANISM_CALIBRATION_SUMMARY_PATH = STAGE2_MECHANISM_CALIBRATION_RUN_ROOT / 'artifacts' / 'stage2_mechanism_calibration_summary.json'
ATTACK_MATRIX_PATH = Path('configs/attacks/real_video_attack_matrix.json')
ABLATION_CONFIG_PATH = Path('configs/ablation/real_video_vae_latent_ablation.json')
MODEL_ID = os.environ.get('TSTW_MODEL_ID', 'stabilityai/sd-vae-ft-mse')
ALLOW_INCONCLUSIVE_REVIEW = os.environ.get('TSTW_ALLOW_INCONCLUSIVE_REVIEW', '0') == '1'
# REQUIRE_FORMAL_PASS = True
REQUIRE_FORMAL_PASS = not ALLOW_INCONCLUSIVE_REVIEW
if ALLOW_INCONCLUSIVE_REVIEW:
    # 仅用于审查 INCONCLUSIVE 结果，不代表 formal PASS 或阶段推进。
    print('ALLOW_INCONCLUSIVE_REVIEW is enabled; formal checker result will be recorded but packaging review may proceed.')
EFFECTIVE_RUNTIME_PROFILE_PLAN = dict(EXECUTION_RUNTIME_PROFILE_CONFIG)
EFFECTIVE_RUNTIME_PROFILE_PLAN['vae_batch_size_frames'] = BATCH_SIZE_FRAMES
EFFECTIVE_RUNTIME_PROFILE_PLAN['batch_size_frames'] = BATCH_SIZE_FRAMES
EFFECTIVE_RUNTIME_PROFILE_PLAN['shard_count'] = SHARD_COUNT
EFFECTIVE_RUNTIME_PROFILE_PLAN['worker_count'] = WORKER_COUNT
EFFECTIVE_RUNTIME_PROFILE_PLAN['cross_event_vae_batching_enabled'] = CROSS_EVENT_VAE_BATCHING_ENABLED
EFFECTIVE_RUNTIME_PROFILE_PLAN['cross_event_vae_decode_batch_size'] = CROSS_EVENT_VAE_DECODE_BATCH_SIZE
EFFECTIVE_RUNTIME_PROFILE_PLAN['cross_event_vae_encode_batch_size'] = CROSS_EVENT_VAE_ENCODE_BATCH_SIZE
RUNTIME_PROFILE_PLAN = runtime_profile_workflow.persist_runtime_profile_plan(
    run_root=RUN_ROOT,
    runtime_profile_payload=EFFECTIVE_RUNTIME_PROFILE_PLAN,
)
run_timer = run_timing_workflow.start_run_timing(
    run_root=RUN_ROOT,
    run_id=RUN_ID,
)
stage2_mechanism_calibration_summary = {'skipped': True}
print({
    'repo_url': REPO_URL,
    'repo_branch': REPO_BRANCH,
    'repo_root': str(REPO_ROOT),
    'notebook_run_preset': NOTEBOOK_RUN_PRESET,
    'run_mode': RUN_MODE,
    'runtime_profile': RUNTIME_PROFILE,
    'execution_runtime_profile': EXECUTION_RUNTIME_PROFILE,
    'batch_size_frames': BATCH_SIZE_FRAMES,
    'shard_count': SHARD_COUNT,
    'shard_index': SHARD_INDEX,
    'worker_count': WORKER_COUNT,
    'cross_event_vae_batching_enabled': CROSS_EVENT_VAE_BATCHING_ENABLED,
    'cross_event_vae_decode_batch_size': CROSS_EVENT_VAE_DECODE_BATCH_SIZE,
    'cross_event_vae_encode_batch_size': CROSS_EVENT_VAE_ENCODE_BATCH_SIZE,
    'runner_samples_per_role_override': RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
    'allow_inconclusive_review': ALLOW_INCONCLUSIVE_REVIEW,
    'require_lpips_evidence': REQUIRE_LPIPS_EVIDENCE,
    'enable_stage2_mechanism_calibration': ENABLE_STAGE2_MECHANISM_CALIBRATION,
    'run_root': str(RUN_ROOT),
    'family_root': str(FAMILY_ROOT),
    'stage2_mechanism_calibration_run_root': str(STAGE2_MECHANISM_CALIBRATION_RUN_ROOT),
    'stage2_mechanism_candidate_method_config_path': str(STAGE2_MECHANISM_CANDIDATE_METHOD_CONFIG_PATH),
})


In [ ]:
STAGE2_CALIBRATION_TARGET = (
    os.environ.get('TSTW_STAGE2_CALIBRATION_TARGET', 'default_search').strip()
    or 'default_search'
 )
if STAGE2_CALIBRATION_TARGET not in {'default_search', 'tl02_controlled_validation'}:
    raise ValueError(
        'TSTW_STAGE2_CALIBRATION_TARGET must be one of: default_search, tl02_controlled_validation'
    )

STAGE2_MECHANISM_CALIBRATION_GRID_BASE_PATH = Path(
    'configs/ablation/stage2_vae_mechanism_calibration_grid.json'
 )
STAGE2_MECHANISM_CALIBRATION_GRID_PATH = STAGE2_MECHANISM_CALIBRATION_GRID_BASE_PATH
TL02_CONTROLLED_EMBEDDING_MARGINS = [0.25, 0.4, 0.6]
TL02_CONTROLLED_SYNC_WIDE_GRID = {
    'fusion_rule': ['sync_rescue_fusion'],
    'lambda_sync': [0.0, 0.025],
    'sync_search_radius': [8, 12],
    'min_sync_positive_margin': [0.0, 0.12],
    'min_sync_alignment_coverage_ratio': [0.25, 0.5],
    'min_sync_alignment_matched_count': [1],
    'min_sync_candidate_score': [0.0],
}
TL02_CONTROLLED_SEARCH_STAGE_NAMES = [
    'anchor_tubelet_only_wide',
    'sync_wide_scan',
]
stage2_controlled_anchor_summary = {
    'mode': 'default_search',
    'tubelet_length': None,
    'spatial_patch_size': None,
    'embedding_projection_support_weight': None,
    'embedding_margin': None,
}
stage2_controlled_search_stage_summary = {
    'search_stage_names': None,
    'anchor_tubelet_only_wide': None,
    'sync_wide_scan': None,
    'sync_refine_scan': None,
}

if STAGE2_CALIBRATION_TARGET == 'tl02_controlled_validation':
    stage2_grid_override_path = (
        STAGE2_MECHANISM_CALIBRATION_RUN_ROOT
        / 'artifacts'
        / 'stage2_vae_mechanism_calibration_grid__tl02_controlled_validation.json'
    )
    stage2_grid_override_path.parent.mkdir(parents=True, exist_ok=True)
    stage2_grid_override_payload = json.loads(
        STAGE2_MECHANISM_CALIBRATION_GRID_BASE_PATH.read_text(encoding='utf-8')
    )
    stage2_grid_override_payload['grid']['tubelet_length'] = [2]
    stage2_grid_override_payload['grid']['spatial_patch_size'] = [[4, 4]]
    stage2_grid_override_payload['grid']['embedding_projection_support_weight'] = [0.25]
    stage2_grid_override_payload['grid']['embedding_margin'] = TL02_CONTROLLED_EMBEDDING_MARGINS
    controlled_search_stages = []
    for stage_payload in stage2_grid_override_payload.get('search_stages', []):
        stage_name = stage_payload.get('stage_name')
        if stage_name == 'anchor_tubelet_only_wide':
            stage_payload.setdefault('grid', {})
            stage_payload['grid']['tubelet_length'] = [2]
            stage_payload['grid']['spatial_patch_size'] = [[4, 4]]
            stage_payload['grid']['embedding_projection_support_weight'] = [0.25]
            stage_payload['grid']['embedding_margin'] = TL02_CONTROLLED_EMBEDDING_MARGINS
            controlled_search_stages.append(stage_payload)
            continue
        if stage_name == 'sync_wide_scan':
            stage_payload.setdefault('grid', {})
            stage_payload['grid'].pop('embedding_margin', None)
            stage_payload['grid'] = {
                **stage_payload['grid'],
                **TL02_CONTROLLED_SYNC_WIDE_GRID,
            }
            controlled_search_stages.append(stage_payload)
    stage2_grid_override_payload['search_stages'] = controlled_search_stages
    stage2_grid_override_path.write_text(
        json.dumps(stage2_grid_override_payload, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    STAGE2_MECHANISM_CALIBRATION_GRID_PATH = stage2_grid_override_path
    stage2_controlled_anchor_summary = {
        'mode': 'tl02_controlled_validation',
        'tubelet_length': 2,
        'spatial_patch_size': [4, 4],
        'embedding_projection_support_weight': 0.25,
        'embedding_margin': TL02_CONTROLLED_EMBEDDING_MARGINS,
    }
    stage2_controlled_search_stage_summary = {
        'search_stage_names': TL02_CONTROLLED_SEARCH_STAGE_NAMES,
        'anchor_tubelet_only_wide': {
            'tubelet_length': [2],
            'spatial_patch_size': [[4, 4]],
            'embedding_projection_support_weight': [0.25],
            'embedding_margin': TL02_CONTROLLED_EMBEDDING_MARGINS,
        },
        'sync_wide_scan': {
            **TL02_CONTROLLED_SYNC_WIDE_GRID,
            'embedding_margin': 'inherit_selected_anchor',
        },
        'sync_refine_scan': {
            'skipped': True,
            'reason': 'tl02_controlled_validation_limits_sync_search_to_anchor_and_narrow_sync_wide',
        },
    }

print({
    'stage2_calibration_target': STAGE2_CALIBRATION_TARGET,
    'stage2_mechanism_calibration_grid_base_path': str(
        STAGE2_MECHANISM_CALIBRATION_GRID_BASE_PATH
    ),
    'stage2_mechanism_calibration_grid_path': str(
        STAGE2_MECHANISM_CALIBRATION_GRID_PATH
    ),
    'stage2_controlled_anchor_summary': stage2_controlled_anchor_summary,
    'stage2_controlled_search_stage_summary': stage2_controlled_search_stage_summary,
})

In [ ]:
ENABLE_STAGE2_LOCAL_CLIP_SYNC_FORENSICS = _resolve_bool_environment_value(
    'TSTW_RUN_STAGE2_LOCAL_CLIP_SYNC_FORENSICS',
    False,
)
PACKAGE_NON_FORMAL_AUDIT_BUNDLE = _resolve_bool_environment_value(
    'TSTW_PACKAGE_NON_FORMAL_AUDIT_BUNDLE',
    False,
)
NON_FORMAL_AUDIT_BUNDLE_NAME = 'real_video_vae_latent_probe_non_formal_audit'
NON_FORMAL_AUDIT_BUNDLE_ROOT = (
    FAMILY_ROOT / 'audit_bundles' / NON_FORMAL_AUDIT_BUNDLE_NAME
)
NON_FORMAL_AUDIT_BUNDLE_ARCHIVE_PATH = (
    NON_FORMAL_AUDIT_BUNDLE_ROOT
    / 'packages'
    / f'{NON_FORMAL_AUDIT_BUNDLE_NAME}.zip'
)
NON_FORMAL_AUDIT_BUNDLE_SUMMARY_PATH = (
    NON_FORMAL_AUDIT_BUNDLE_ROOT / 'audit_bundle_summary.json'
)
STAGE2_LOCAL_CLIP_SYNC_DIAGNOSTICS_PATH = (
    STAGE2_MECHANISM_CALIBRATION_RUN_ROOT
    / 'artifacts'
    / 'selected_candidate_local_clip_sync_diagnostics.csv'
)
stage2_local_clip_sync_diagnostics_summary = {'skipped': True, 'reason': 'not_run_yet'}
non_formal_audit_bundle_summary = {'skipped': True, 'reason': 'not_run_yet'}
print({
    'enable_stage2_local_clip_sync_forensics': ENABLE_STAGE2_LOCAL_CLIP_SYNC_FORENSICS,
    'package_non_formal_audit_bundle': PACKAGE_NON_FORMAL_AUDIT_BUNDLE,
    'stage2_local_clip_sync_diagnostics_path': str(STAGE2_LOCAL_CLIP_SYNC_DIAGNOSTICS_PATH),
    'non_formal_audit_bundle_archive_path': str(NON_FORMAL_AUDIT_BUNDLE_ARCHIVE_PATH),
    'non_formal_audit_bundle_summary_path': str(NON_FORMAL_AUDIT_BUNDLE_SUMMARY_PATH),
})

## 01 Mount Google Drive


In [ ]:
import importlib

try:
    colab_drive = importlib.import_module('google.colab.drive')
except ModuleNotFoundError as exc:
    raise RuntimeError('This notebook entrypoint must be executed inside Google Colab.') from exc
colab_drive.mount('/content/drive')

## 02 Read Processed Dataset Registry

确认上游 processed dataset 根目录存在, 并打印当前 `PROCESSED_DATASET_KEY` 与 Drive 侧 processed dataset 路径。

In [ ]:
if not PROCESSED_DATASET_ROOT.exists():
    raise FileNotFoundError(PROCESSED_DATASET_ROOT)
print({
    'processed_dataset_root': str(PROCESSED_DATASET_ROOT),
    'processed_dataset_key': PROCESSED_DATASET_KEY,
})

## 03 Prepare Local Runtime Workspace

复制 processed dataset 到 local runtime, 创建 run root 与 family root, 并解析本次 runner 必须使用的 session local `dataset_manifest.json`。

In [ ]:
runtime_workspace_handoff = probe_workflow.prepare_probe_runtime_workspace(
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    local_dataset_root=LOCAL_DATASET_ROOT,
    family_root=FAMILY_ROOT,
    run_root=RUN_ROOT,
    copy_processed_dataset=True,
)
LOCAL_DATASET_ROOT = Path(runtime_workspace_handoff['local_dataset_root'])
PROCESSED_DATASET_MANIFEST = Path(runtime_workspace_handoff['local_dataset_manifest_path'])
if not PROCESSED_DATASET_MANIFEST.exists():
    raise FileNotFoundError(PROCESSED_DATASET_MANIFEST)
print(runtime_workspace_handoff)

## 04 Clone Or Update Repository

记录当前仓库 short commit, 设置 `PYTHONPATH`, 并打印 git 工作区状态, 便于结果包追踪到具体代码版本。

In [ ]:
repo_env = dict(os.environ)
repo_env['PYTHONPATH'] = str(REPO_ROOT)
GIT_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'],
    text=True,
    env=repo_env,
    cwd=REPO_ROOT,
).strip()
subprocess.run(['git', 'status', '--short'], check=True, env=repo_env, cwd=REPO_ROOT)

## 05 Install Runtime Dependencies

安装运行所需依赖, 包括测试工具、diffusers 相关依赖、模型加载依赖和视频质量指标依赖。

In [ ]:
subprocess.run(
    ['apt-get', 'update'],
    check=True,
)
subprocess.run(
    ['apt-get', 'install', '-y', 'ffmpeg', 'zstd'],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        'pytest',
        'diffusers',
        'accelerate',
        'transformers',
        'safetensors',
        'huggingface_hub',
        'lpips',
        'pytorch-msssim',
        'zstandard',
        'opencv-python-headless',
        'scikit-image',
        'imageio-ffmpeg',
    ],
    check=True,
    env=repo_env,
    cwd=REPO_ROOT,
)

## 06 Prepare Session Model

准备 session-only AutoencoderKL 模型目录, 并写出 session model manifest。

In [ ]:
with run_timer.event('model_preparation', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
    session_model_manifest = probe_workflow.prepare_probe_session_model(
        model_id=MODEL_ID,
        local_model_root=LOCAL_MODEL_ROOT,
        session_manifest_path=SESSION_MODEL_MANIFEST_PATH,
        revision='main',
    )
    LOCAL_MODEL_ROOT = Path(session_model_manifest['models'][0]['local_path'])
    if not LOCAL_MODEL_ROOT.exists():
        raise FileNotFoundError(LOCAL_MODEL_ROOT)
    assert session_model_manifest['model_policy'] == 'session_only_no_drive_model_storage'
if ENABLE_LPIPS:
    LPIPS_MODEL_ROOT_PATH.mkdir(parents=True, exist_ok=True)
    LPIPS_CACHE_MARKER_PATH.write_text(
        'session_local_lpips_cache_root\n',
        encoding='utf-8',
    )
    os.environ['TSTW_LPIPS_MODEL_ROOT'] = str(LPIPS_MODEL_ROOT_PATH)
    os.environ['LPIPS_MODEL_ROOT'] = str(LPIPS_MODEL_ROOT_PATH)
    os.environ['TORCH_HOME'] = str(LPIPS_MODEL_ROOT_PATH)
if PROFILE_DRIVE_IO:
    drive_io_profile = runtime_profile_workflow.profile_drive_io(
        run_root=RUN_ROOT,
        drive_root=DRIVE_ROOT,
        local_root=LOCAL_RUNTIME_ROOT,
        sample_size_mb=DRIVE_IO_SAMPLE_SIZE_MB,
    )
else:
    drive_io_profile = {'skipped': True}
print(session_model_manifest)
print({
    'drive_io_profile': drive_io_profile,
    'lpips_model_root': str(LPIPS_MODEL_ROOT_PATH) if ENABLE_LPIPS else None,
    'lpips_cache_marker_path': str(LPIPS_CACHE_MARKER_PATH) if ENABLE_LPIPS else None,
})

## 07 Write Runtime Config

输出 `RUNTIME_CONFIG_PATH` 是后续 runner、runtime manifest 与结果包审计的核心输入。


In [ ]:
runtime_extra_config = {
    'family_id': FAMILY_ID,
    'workflow_key': WORKFLOW_KEY,
    'step_key': STEP_KEY,
    'git_commit': GIT_COMMIT,
    'notebook_run_preset': NOTEBOOK_RUN_PRESET,
    'execution_runtime_profile': EXECUTION_RUNTIME_PROFILE,
    'runtime_profile_config_path': EFFECTIVE_RUNTIME_PROFILE_PLAN['config_path'],
    'runtime_profile_config_digest': EFFECTIVE_RUNTIME_PROFILE_PLAN['config_digest'],
    'batch_size_frames': BATCH_SIZE_FRAMES,
    'cross_event_vae_batching_enabled': CROSS_EVENT_VAE_BATCHING_ENABLED,
    'cross_event_vae_decode_batch_size': CROSS_EVENT_VAE_DECODE_BATCH_SIZE,
    'cross_event_vae_encode_batch_size': CROSS_EVENT_VAE_ENCODE_BATCH_SIZE,
    'lpips_batch_size': LPIPS_BATCH_SIZE,
    'clip_batch_size': CLIP_BATCH_SIZE,
    'worker_count': WORKER_COUNT,
    'video_io_worker_count': VIDEO_IO_WORKER_COUNT,
    'attack_worker_count': ATTACK_WORKER_COUNT,
    'shard_count': SHARD_COUNT,
    'shard_index': SHARD_INDEX,
    'reuse_encoded_latents': REUSE_ENCODED_LATENTS,
    'reuse_decoded_videos': REUSE_DECODED_VIDEOS,
    'reuse_attacked_videos': REUSE_ATTACKED_VIDEOS,
    'quality_metrics': {
        'enable_lpips': ENABLE_LPIPS,
        'enable_clip_similarity': ENABLE_CLIP_SIMILARITY,
    },
    'temporal_metrics': {
        'enable_motion_consistency': ENABLE_MOTION_CONSISTENCY,
    },
    'runtime_manifest_overrides': {
        'family_root': str(FAMILY_ROOT),
        'session_model_manifest_path': str(SESSION_MODEL_MANIFEST_PATH),
    },
}
if ENABLE_LPIPS:
    runtime_extra_config['local_lpips_model_root'] = str(LPIPS_MODEL_ROOT_PATH)
runtime_config_handoff = probe_workflow.write_probe_runtime_config(
    runtime_config_path=RUNTIME_CONFIG_PATH,
    execution_environment='colab',
    processed_dataset_key=PROCESSED_DATASET_KEY,
    local_dataset_root=LOCAL_DATASET_ROOT,
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    vae_model_local_path=LOCAL_MODEL_ROOT,
    dataset_manifest_path=PROCESSED_DATASET_MANIFEST,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
    extra_config=runtime_extra_config,
)
print(runtime_config_handoff)
print({
    'execution_runtime_profile': EXECUTION_RUNTIME_PROFILE,
    'runtime_profile_plan': RUNTIME_PROFILE_PLAN,
    'batch_size_frames': BATCH_SIZE_FRAMES,
    'lpips_batch_size': LPIPS_BATCH_SIZE,
    'clip_batch_size': CLIP_BATCH_SIZE,
    'worker_count': WORKER_COUNT,
    'cross_event_vae_batching_enabled': CROSS_EVENT_VAE_BATCHING_ENABLED,
    'cross_event_vae_decode_batch_size': CROSS_EVENT_VAE_DECODE_BATCH_SIZE,
    'cross_event_vae_encode_batch_size': CROSS_EVENT_VAE_ENCODE_BATCH_SIZE,
    'shard_count': SHARD_COUNT,
    'shard_index': SHARD_INDEX,
    'require_lpips_evidence': REQUIRE_LPIPS_EVIDENCE,
    'lpips_enabled': ENABLE_LPIPS,
    'lpips_model_root': str(LPIPS_MODEL_ROOT_PATH) if ENABLE_LPIPS else None,
    'runner_samples_per_role_override': RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
})
if PROFILE_RUNTIME:
    runtime_environment_snapshot = runtime_profile_workflow.capture_colab_environment(
        run_root=RUN_ROOT,
        run_id=RUN_ID,
        run_mode=RUN_MODE,
        runtime_profile=RUNTIME_PROFILE,
    )
    run_scale_estimate = runtime_profile_workflow.estimate_real_video_vae_latent_run_scale(
        run_root=RUN_ROOT,
        dataset_manifest=PROCESSED_DATASET_MANIFEST,
        attack_matrix=ATTACK_MATRIX_PATH,
        ablation_config=ABLATION_CONFIG_PATH,
        protocol_config=PROTOCOL_CONFIG_PATH,
        runtime_profile=RUNTIME_PROFILE,
        samples_per_role=RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
        batch_size_frames=BATCH_SIZE_FRAMES,
    )
else:
    runtime_environment_snapshot = {'skipped': True}
    run_scale_estimate = {'skipped': True}
print(runtime_environment_snapshot)
print(run_scale_estimate)


## 08 Check GPU And Runtime

作用：运行 runtime preflight, 检查当前运行模式、processed dataset 本地目录和 session model 目录是否满足运行前置条件。

In [ ]:
with run_timer.event('runtime_preflight', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
    runtime_check_report = probe_workflow.run_probe_runtime_preflight(
        run_mode=RUN_MODE,
        local_dataset_root=LOCAL_DATASET_ROOT,
        local_model_dirs=[LOCAL_MODEL_ROOT, LPIPS_MODEL_ROOT_PATH] if ENABLE_LPIPS else [LOCAL_MODEL_ROOT],
    )
print(runtime_check_report)

## 09 Verify Repository Contract

项目合同校验与总审计。


In [ ]:
subprocess.run([sys.executable, 'tools/harness/validate_project_contract.py'], check=True, env=repo_env, cwd=REPO_ROOT)
subprocess.run([sys.executable, 'tools/harness/run_all_audits.py'], check=True, env=repo_env, cwd=REPO_ROOT)

## 10 Run Smoke Tests

执行真实视频 VAE encode/decode smoke 测试, 判断视频读写、VAE metadata 与 digest 稳定性。

In [ ]:
subprocess.run(
    [
        sys.executable,
        '-m',
        'pytest',
        '-q',
        '-m',
        'smoke',
        'tests/integration/test_real_video_vae_encode_decode_smoke.py',
    ],
    check=True,
    env=repo_env,
    cwd=REPO_ROOT,
)

## 11 Run Real Video VAE Latent Formal

启动 probe runner, 由仓库模块生成 records、thresholds、manifests、artifacts 与后续可重建表格所需的证据。

`run_root=RUN_ROOT` 指定 session local 运行目录；`runtime_config_path=RUNTIME_CONFIG_PATH` 提供运行配置；`dataset_manifest=PROCESSED_DATASET_MANIFEST` 显式传入 processed dataset manifest, 防止回退到默认 manifest。

In [ ]:
import importlib
probe_workflow = importlib.reload(probe_workflow)
runtime_profile_workflow = importlib.reload(runtime_profile_workflow)
run_timing_workflow = importlib.reload(run_timing_workflow)
RUN_MAIN_FORMAL = os.environ.get('TSTW_RUN_MAIN_FORMAL', '1') == '1'
gpu_profile_process = None
runner_error = None
if RUN_MAIN_FORMAL:
    try:
        if PROFILE_GPU_RUNTIME:
            gpu_profile_process = runtime_profile_workflow.start_gpu_runtime_profile(
                run_root=RUN_ROOT,
                interval_seconds=GPU_PROFILE_INTERVAL_SECONDS,
            )
        with run_timer.event('real_video_vae_latent_runner', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
            probe_workflow.run_probe_runner(
                run_root=RUN_ROOT,
                run_mode=RUN_MODE,
                runtime_profile=RUNTIME_PROFILE,
                runtime_config_path=RUNTIME_CONFIG_PATH,
                attack_matrix=ATTACK_MATRIX_PATH,
                ablation_config=ABLATION_CONFIG_PATH,
                dataset_manifest=PROCESSED_DATASET_MANIFEST,
                samples_per_role=RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
                batch_size_frames=BATCH_SIZE_FRAMES,
                shard_count=SHARD_COUNT,
                shard_index=SHARD_INDEX,
                worker_count=WORKER_COUNT,
                cross_event_vae_batching_enabled=CROSS_EVENT_VAE_BATCHING_ENABLED,
                cross_event_vae_decode_batch_size=CROSS_EVENT_VAE_DECODE_BATCH_SIZE,
                cross_event_vae_encode_batch_size=CROSS_EVENT_VAE_ENCODE_BATCH_SIZE,
                python_executable=sys.executable,
            )
    except Exception as error:
        runner_error = error
    finally:
        if PROFILE_GPU_RUNTIME:
            runtime_profile_workflow.stop_gpu_runtime_profile(
                gpu_profile_process,
                run_root=RUN_ROOT,
            )
    gpu_runtime_summary = runtime_profile_workflow.summarize_gpu_runtime_profile(
        run_root=RUN_ROOT,
    )
    run_progress_snapshot = runtime_profile_workflow.watch_real_video_vae_latent_progress(
        run_root=RUN_ROOT,
    )
    run_failure_summary = runtime_profile_workflow.summarize_run_failures(
        run_root=RUN_ROOT,
    )
    if WRITE_RUNTIME_RECOMMENDATION:
        runtime_parameter_recommendation = runtime_profile_workflow.recommend_runtime_parameters(
            run_root=RUN_ROOT,
        )
    else:
        runtime_parameter_recommendation = {'skipped': True}
else:
    gpu_runtime_summary = {'skipped': True, 'reason': 'run_main_formal_disabled'}
    run_progress_snapshot = {'skipped': True, 'reason': 'run_main_formal_disabled'}
    run_failure_summary = {'skipped': True, 'reason': 'run_main_formal_disabled'}
    runtime_parameter_recommendation = {'skipped': True, 'reason': 'run_main_formal_disabled'}
    print({'main_formal_skipped': True, 'run_main_formal': RUN_MAIN_FORMAL})
cross_event_vae_batching_summary_path = RUNTIME_PROFILE_ROOT / 'cross_event_vae_batching_summary.json'
if cross_event_vae_batching_summary_path.exists():
    cross_event_vae_batching_summary = json.loads(cross_event_vae_batching_summary_path.read_text(encoding='utf-8'))
else:
    cross_event_vae_batching_summary = {'enabled': False, 'reason': 'cross_event_vae_batching_disabled'}
run_timing_summary = run_timing_workflow.summarize_run_timing(
    run_root=RUN_ROOT,
)
print(gpu_runtime_summary)
print(cross_event_vae_batching_summary)
print(run_timing_summary)
print(run_progress_snapshot)
print(run_failure_summary)
print(runtime_parameter_recommendation)
if runner_error is not None:
    raise runner_error


## 12 Rebuild Tables And Reports

records 与 manifests 重建 tables、reports、figures 和 failure gallery 等派生产物。

In [ ]:
RUN_MAIN_FORMAL = os.environ.get('TSTW_RUN_MAIN_FORMAL', '1') == '1'
if RUN_MAIN_FORMAL:
    with run_timer.event('table_and_report_rebuild', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
        probe_workflow.rebuild_probe_tables_and_reports(run_root=RUN_ROOT)
else:
    print({'table_and_report_rebuild_skipped': True, 'reason': 'run_main_formal_disabled'})

## 13 Check Real Video VAE Latent Outputs

校验输出完整性、formal pass 条件、runtime manifest、records-to-artifacts 链路与下一阶段门禁。

In [ ]:
RUN_MAIN_FORMAL = os.environ.get('TSTW_RUN_MAIN_FORMAL', '1') == '1'
formal_validation_summary = {'skipped': True, 'reason': 'run_main_formal_disabled'}
formal_checker_error = None
if RUN_MAIN_FORMAL:
    try:
        with run_timer.event('formal_checker', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
            formal_validation_summary = probe_workflow.check_probe_outputs(
                run_root=RUN_ROOT,
                construction_phase='real_video_vae_latent_probe',
                run_mode=RUN_MODE,
                require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
            )
            RUNTIME_PROFILE_ROOT.mkdir(parents=True, exist_ok=True)
            FORMAL_VALIDATION_SUMMARY_PATH.write_text(
                json.dumps(formal_validation_summary, ensure_ascii=False, indent=2) + '\n',
                encoding='utf-8',
            )
            if not formal_validation_summary['status']:
                raise RuntimeError(formal_validation_summary)
    except RuntimeError as error:
        formal_checker_error = error
else:
    print(formal_validation_summary)

run_timing_summary = run_timing_workflow.summarize_run_timing(
    run_root=RUN_ROOT,
)

run_failure_summary = runtime_profile_workflow.summarize_run_failures(
    run_root=RUN_ROOT,
)
if WRITE_RUNTIME_RECOMMENDATION:
    runtime_parameter_recommendation = runtime_profile_workflow.recommend_runtime_parameters(
        run_root=RUN_ROOT,
    )
else:
    runtime_parameter_recommendation = {'skipped': True}

print(formal_validation_summary)
print(run_timing_summary)
print(run_failure_summary)
print(runtime_parameter_recommendation)
if formal_checker_error is not None:
    raise formal_checker_error

## 14 Run Stage2 Mechanism Audit

基于既有 records、thresholds 与 tables 重建阶段 2 mechanism audit，区分 implementation completion 与 mechanism evidence；若设置 TSTW_RUN_STAGE2_MECHANISM_CALIBRATION=1，则在同一步骤额外启动独立 calibration run，生成受治理的 tubelet_sync candidate method config。

`mechanism_config_path=STAGE2_MECHANISM_CONFIG_PATH` 控制门禁阈值；
`stage2_mechanism_summary` 会写回 run_root 并继续传给 family packaging；
`stage2_mechanism_calibration_summary` 使用独立 run root，避免覆盖 formal 主运行产物。

In [ ]:
RUN_MAIN_FORMAL = os.environ.get('TSTW_RUN_MAIN_FORMAL', '1') == '1'
if RUN_MAIN_FORMAL:
    with run_timer.event('stage2_mechanism_audit', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
        stage2_mechanism_summary = probe_workflow.run_probe_stage2_mechanism_audit(
            run_root=RUN_ROOT,
            mechanism_config_path=STAGE2_MECHANISM_CONFIG_PATH,
        )
    event_scores_path = RUN_ROOT / 'records' / 'event_scores.jsonl'
    if not event_scores_path.exists():
        raise FileNotFoundError(event_scores_path)
    lpips_evidence_summary = {
        'lpips_required': REQUIRE_LPIPS_EVIDENCE,
        'lpips_model_root': LPIPS_MODEL_ROOT or None,
        'lpips_record_count': 0,
        'lpips_failure_count': 0,
        'samples_per_role_override': RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
    }
    with event_scores_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            record = json.loads(line)
            quality_metrics = record.get('quality_metrics', {})
            if not isinstance(quality_metrics, dict):
                continue
            if quality_metrics.get('watermarked_video_lpips') is not None:
                lpips_evidence_summary['lpips_record_count'] += 1
            lpips_failure_reason = quality_metrics.get('lpips_failure_reason')
            if lpips_failure_reason not in {None, 'lpips_disabled_by_config'}:
                lpips_evidence_summary['lpips_failure_count'] += 1
else:
    stage2_mechanism_summary = {'skipped': True, 'reason': 'run_main_formal_disabled'}
    lpips_evidence_summary = {'skipped': True, 'reason': 'run_main_formal_disabled'}
stage2_calibration_gpu_profile_enabled = (
    not RUN_MAIN_FORMAL
    and ENABLE_STAGE2_MECHANISM_CALIBRATION
    and PROFILE_GPU_RUNTIME
    )
stage2_calibration_gpu_profile_process = None
if ENABLE_STAGE2_MECHANISM_CALIBRATION:
    try:
        if stage2_calibration_gpu_profile_enabled:
            stage2_calibration_gpu_profile_process = runtime_profile_workflow.start_gpu_runtime_profile(
                run_root=STAGE2_MECHANISM_CALIBRATION_RUN_ROOT,
                interval_seconds=GPU_PROFILE_INTERVAL_SECONDS,
            )
        with run_timer.event('stage2_mechanism_calibration', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
            stage2_mechanism_calibration_summary = probe_workflow.run_probe_stage2_mechanism_calibration(
                run_root=STAGE2_MECHANISM_CALIBRATION_RUN_ROOT,
                run_mode=RUN_MODE,
                runtime_profile=RUNTIME_PROFILE,
                grid_config_path=STAGE2_MECHANISM_CALIBRATION_GRID_PATH,
                attack_matrix_path=ATTACK_MATRIX_PATH,
                ablation_config_path=ABLATION_CONFIG_PATH,
                mechanism_config_path=STAGE2_MECHANISM_CONFIG_PATH,
                dataset_manifest_path=PROCESSED_DATASET_MANIFEST,
                runtime_config_path=RUNTIME_CONFIG_PATH,
                samples_per_role=RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
                batch_size_frames=BATCH_SIZE_FRAMES,
                output_method_config_path=STAGE2_MECHANISM_CANDIDATE_METHOD_CONFIG_PATH,
            )
        if ENABLE_STAGE2_LOCAL_CLIP_SYNC_FORENSICS:
            with run_timer.event('stage2_local_clip_sync_diagnostics', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
                stage2_local_clip_sync_diagnostics_summary = probe_workflow.write_probe_stage2_local_clip_sync_diagnostics(
                    run_root=STAGE2_MECHANISM_CALIBRATION_RUN_ROOT,
                    calibration_summary=stage2_mechanism_calibration_summary,
                    output_csv_path=STAGE2_LOCAL_CLIP_SYNC_DIAGNOSTICS_PATH,
                )
        else:
            stage2_local_clip_sync_diagnostics_summary = {
                'skipped': True,
                'reason': 'stage2_local_clip_sync_forensics_disabled',
            }
    finally:
        if stage2_calibration_gpu_profile_enabled:
            runtime_profile_workflow.stop_gpu_runtime_profile(
                stage2_calibration_gpu_profile_process,
                run_root=STAGE2_MECHANISM_CALIBRATION_RUN_ROOT,
            )
    if stage2_calibration_gpu_profile_enabled:
        gpu_runtime_summary = runtime_profile_workflow.summarize_gpu_runtime_profile(
            run_root=STAGE2_MECHANISM_CALIBRATION_RUN_ROOT,
        )
else:
    stage2_mechanism_calibration_summary = {'skipped': True}
    stage2_local_clip_sync_diagnostics_summary = {
        'skipped': True,
        'reason': 'stage2_mechanism_calibration_disabled',
    }
print(stage2_mechanism_summary)
print(lpips_evidence_summary)
print(stage2_mechanism_calibration_summary)
print(stage2_local_clip_sync_diagnostics_summary)
print({
    'stage2_mechanism_summary_path': str(STAGE2_MECHANISM_SUMMARY_PATH),
    'stage2_mechanism_calibration_summary_path': str(STAGE2_MECHANISM_CALIBRATION_SUMMARY_PATH),
    'stage2_mechanism_candidate_method_config_path': str(STAGE2_MECHANISM_CANDIDATE_METHOD_CONFIG_PATH),
    'stage2_local_clip_sync_diagnostics_path': str(STAGE2_LOCAL_CLIP_SYNC_DIAGNOSTICS_PATH),
    'enable_stage2_mechanism_calibration': ENABLE_STAGE2_MECHANISM_CALIBRATION,
    'enable_stage2_local_clip_sync_forensics': ENABLE_STAGE2_LOCAL_CLIP_SYNC_FORENSICS,
    'require_stage2_mechanism_pass': REQUIRE_STAGE2_MECHANISM_PASS,
    'run_main_formal': RUN_MAIN_FORMAL,
    'stage2_calibration_gpu_profile_enabled': stage2_calibration_gpu_profile_enabled,
})
if RUN_MAIN_FORMAL and REQUIRE_LPIPS_EVIDENCE and lpips_evidence_summary['lpips_record_count'] == 0:
    raise RuntimeError(lpips_evidence_summary)
if RUN_MAIN_FORMAL and REQUIRE_STAGE2_MECHANISM_PASS and stage2_mechanism_summary['Stage2MechanismDecision'] != 'PASS':
    raise RuntimeError(stage2_mechanism_summary)

## 15 Package Family Results

生成 family 结果包、登记 registry, 并打印最终摘要。

`formal_validation_summary` 作为 implementation gate 证据；
`stage2_mechanism_summary` 作为 mechanism gate 证据并写入 family summary。

In [ ]:
RUN_MAIN_FORMAL = os.environ.get('TSTW_RUN_MAIN_FORMAL', '1') == '1'
gpu_runtime_trace_path = RUNTIME_PROFILE_ROOT / 'gpu_runtime_trace.csv'
gpu_runtime_audit_record_path = RUNTIME_PROFILE_ROOT / 'gpu_runtime_audit_record.json'
run_timing_summary_path = RUNTIME_PROFILE_ROOT / 'run_timing_summary.json'
summarize_run_timing = run_timing_workflow.summarize_run_timing
gpu_profiling_mode = (
    'formal_runner'
    if RUN_MAIN_FORMAL
    else (
        'stage2_mechanism_calibration'
        if ENABLE_STAGE2_MECHANISM_CALIBRATION
        else 'notebook_setup_only'
    )
)
gpu_profiling_expected = bool(RUN_MAIN_FORMAL or stage2_calibration_gpu_profile_enabled)
run_timing_summary = summarize_run_timing(
    run_root=RUN_ROOT,
)
run_failure_summary = runtime_profile_workflow.summarize_run_failures(
    run_root=RUN_ROOT,
)
gpu_runtime_audit_record = runtime_profile_workflow.write_gpu_runtime_audit_record(
    run_root=RUN_ROOT,
    profiling_mode=gpu_profiling_mode,
    profiling_expected=gpu_profiling_expected,
)
if isinstance(gpu_runtime_audit_record.get('gpu_runtime_summary'), dict):
    gpu_runtime_summary = gpu_runtime_audit_record['gpu_runtime_summary']
if WRITE_RUNTIME_RECOMMENDATION:
    runtime_parameter_recommendation = runtime_profile_workflow.recommend_runtime_parameters(
        run_root=RUN_ROOT,
    )
else:
    runtime_parameter_recommendation = {'skipped': True}
if RUN_MAIN_FORMAL:
    with run_timer.event('result_packaging', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
        package_payload = probe_workflow.package_probe_family_results(
            run_root=RUN_ROOT,
            family_root=FAMILY_ROOT,
            require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
            formal_validation_summary=formal_validation_summary,
            drive_root=DRIVE_ROOT,
            family_id=FAMILY_ID,
            workflow_key=WORKFLOW_KEY,
            step_key=STEP_KEY,
            run_mode=RUN_MODE,
            mechanism_summary=stage2_mechanism_summary,
        )
    drive_archive_path = package_payload['drive_archive_path']
    compat_pack_root = package_payload['compat_pack_root']
    non_formal_audit_bundle_summary = {'skipped': True, 'reason': 'run_main_formal_enabled'}
    stage2_calibration_family_export_summary = {'skipped': True, 'reason': 'run_main_formal_enabled'}
else:
    package_payload = {'skipped': True, 'reason': 'run_main_formal_disabled', 'registry_paths': {}}
    drive_archive_path = None
    compat_pack_root = None
    if ENABLE_STAGE2_MECHANISM_CALIBRATION:
        stage2_calibration_family_export_summary = probe_workflow.export_probe_stage2_calibration_family_snapshot(
            family_root=FAMILY_ROOT,
            calibration_run_root=STAGE2_MECHANISM_CALIBRATION_RUN_ROOT,
            calibration_summary=stage2_mechanism_calibration_summary,
            diagnostics_csv_path=(
                STAGE2_LOCAL_CLIP_SYNC_DIAGNOSTICS_PATH
                if STAGE2_LOCAL_CLIP_SYNC_DIAGNOSTICS_PATH.exists()
                else None
            ),
        )
    else:
        stage2_calibration_family_export_summary = {
            'skipped': True,
            'reason': 'stage2_mechanism_calibration_disabled',
        }
    if PACKAGE_NON_FORMAL_AUDIT_BUNDLE:
        with run_timer.event('audit_bundle_packaging', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
            non_formal_audit_bundle_summary = probe_workflow.package_probe_non_formal_audit_bundle(
                family_root=FAMILY_ROOT,
                notebook_run_root=RUN_ROOT,
                calibration_run_root=(
                    STAGE2_MECHANISM_CALIBRATION_RUN_ROOT
                    if ENABLE_STAGE2_MECHANISM_CALIBRATION
                    else None
                ),
                calibration_summary=(
                    stage2_mechanism_calibration_summary
                    if ENABLE_STAGE2_MECHANISM_CALIBRATION
                    else None
                ),
                diagnostics_csv_path=(
                    STAGE2_LOCAL_CLIP_SYNC_DIAGNOSTICS_PATH
                    if STAGE2_LOCAL_CLIP_SYNC_DIAGNOSTICS_PATH.exists()
                    else None
                ),
                bundle_name=NON_FORMAL_AUDIT_BUNDLE_NAME,
            )
    else:
        non_formal_audit_bundle_summary = {
            'skipped': True,
            'reason': 'package_non_formal_audit_bundle_disabled',
        }
run_timing_summary = run_timing_workflow.summarize_run_timing(
    run_root=RUN_ROOT,
)
final_family_summary = {
    'family_id': FAMILY_ID,
    'run_root': str(RUN_ROOT),
    'family_root': str(FAMILY_ROOT),
    'execution_runtime_profile': EXECUTION_RUNTIME_PROFILE,
    'runtime_profile_plan': RUNTIME_PROFILE_PLAN,
    'runtime_environment_snapshot': runtime_environment_snapshot,
    'drive_io_profile': drive_io_profile,
    'run_scale_estimate': run_scale_estimate,
    'gpu_runtime_summary': gpu_runtime_summary,
    'run_timing_summary': run_timing_summary,
    'run_progress_snapshot': run_progress_snapshot,
    'run_failure_summary': run_failure_summary,
    'runtime_parameter_recommendation': runtime_parameter_recommendation,
    'formal_validation_summary': formal_validation_summary,
    'stage2_mechanism_summary': stage2_mechanism_summary,
    'stage2_mechanism_calibration_summary': stage2_mechanism_calibration_summary,
    'stage2_local_clip_sync_diagnostics_summary': stage2_local_clip_sync_diagnostics_summary,
    'stage2_calibration_family_export_summary': stage2_calibration_family_export_summary,
    'non_formal_audit_bundle_summary': non_formal_audit_bundle_summary,
    'lpips_evidence_summary': lpips_evidence_summary,
    'batch_size_frames': BATCH_SIZE_FRAMES,
    'require_lpips_evidence': REQUIRE_LPIPS_EVIDENCE,
    'runner_samples_per_role_override': RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
    'gpu_runtime_trace_path': str(gpu_runtime_trace_path),
    'gpu_runtime_audit_record_path': str(gpu_runtime_audit_record_path),
    'run_timing_summary_path': str(run_timing_summary_path),
    'stage2_local_clip_sync_diagnostics_path': str(STAGE2_LOCAL_CLIP_SYNC_DIAGNOSTICS_PATH),
    'non_formal_audit_bundle_archive_path': non_formal_audit_bundle_summary.get('archive_path'),
    'non_formal_audit_bundle_summary_path': non_formal_audit_bundle_summary.get('summary_path'),
    'drive_archive_path': None if drive_archive_path is None else str(drive_archive_path),
    'compat_pack_root': None if compat_pack_root is None else str(compat_pack_root),
    'result_registry.jsonl': package_payload.get('registry_paths', {}).get('result_registry.jsonl'),
    'family_registry.jsonl': package_payload.get('registry_paths', {}).get('family_registry.jsonl'),
}
print({
    'drive_archive_path': drive_archive_path,
    'compat_pack_root': compat_pack_root,
    'result_registry.jsonl': package_payload.get('registry_paths', {}).get('result_registry.jsonl'),
    'family_registry.jsonl': package_payload.get('registry_paths', {}).get('family_registry.jsonl'),
    'gpu_runtime_audit_record_path': str(gpu_runtime_audit_record_path),
    'run_timing_summary_path': str(run_timing_summary_path),
    'stage2_local_clip_sync_diagnostics_path': str(STAGE2_LOCAL_CLIP_SYNC_DIAGNOSTICS_PATH),
    'stage2_calibration_family_export_summary': stage2_calibration_family_export_summary,
    'non_formal_audit_bundle_archive_path': non_formal_audit_bundle_summary.get('archive_path'),
    'non_formal_audit_bundle_summary_path': non_formal_audit_bundle_summary.get('summary_path'),
    'run_main_formal': RUN_MAIN_FORMAL,
})
print(run_timing_summary)
print(final_family_summary)